In [ ]:
import pandas as pd
from pathlib import Path

pasta = Path("../data/raw/microdados_enem_2025")

arquivos ={
    "participantes": pasta / "DADOS" / "PARTICIPANTES_2025.csv",
    "resultados": pasta / "DADOS" / "RESULTADOS_2025.csv",
}

for nome, caminho in  arquivos.items():
    amostra = pd.read_csv(caminho, sep=";", encoding="latin-1", nrows=5)
    print(nome, amostra.shape)
    print(list(amostra.columns))
    print()



In [ ]:
achados = list(pasta.glob("*/*.xlsx"))
dic = pd.read_excel(achados[0], sheet_name=None)

for aba, tabela in dic.items():
    print(aba, tabela.shape)

In [ ]:
res = dic["RESULTADOS_2025"]
print(res.columns.tolist())
print(res.head(8).to_string())


In [ ]:
def buscar(tabela, texto):
    achou = tabela.astype(str).apply(
        lambda coluna: coluna.str.contains(texto, case=False, na=False)
    ).any(axis=1)
    return tabela[achou]

print(buscar(res, "NOTA").to_string())

In [ ]:
def mostrar(tabela, variavel, linhas=8):
    posicoes = tabela.index[tabela.iloc[:, 0] == variavel]
    for p in posicoes:
        print(tabela.iloc[p : p + linhas].to_string(header=False))
        print()


for v in ["TP_PRESENCA_CN", "TP_STATUS_REDACAO", "TP_DEPENDENCIA_ADM_ESC",
          "TP_LOCALIZACAO_ESC", "SG_UF_ESC", "SG_UF_PROVA", "CO_ESCOLA"]:
    print("=====", v)
    mostrar(res, v)

print("===== RODAPÉ")
print(res.tail(15).to_string(header=False))

In [ ]:
colunas = [
    "SG_UF_ESC", "SG_UF_PROVA", "TP_DEPENDENCIA_ADM_ESC", "TP_LOCALIZACAO_ESC",
    "TP_PRESENCA_CN", "TP_PRESENCA_CH", "TP_PRESENCA_LC", "TP_PRESENCA_MT",
    "NU_NOTA_CN", "NU_NOTA_CH", "NU_NOTA_LC", "NU_NOTA_MT",
    "TP_STATUS_REDACAO", "NU_NOTA_REDACAO",
]

df = pd.read_csv(arquivos["resultados"], sep=";", encoding="latin-1", usecols=colunas)

print(df.shape)
print(df.dtypes)
print(df.head())

In [ ]:
areas = ["CN", "CH", "LC", "MT"]

for area in areas:
    print("Presença", area)
    print(df[f"TP_PRESENCA_{area}"].value_counts().sort_index())
    print()

print("Status da redação")
print(df["TP_STATUS_REDACAO"].value_counts(dropna=False).sort_index())
print()

print("Proporção sem escola informada:", df["TP_DEPENDENCIA_ADM_ESC"].isna().mean())
print("Linhas sem UF da prova:", df["SG_UF_PROVA"].isna().sum())
print()

for area in areas:
    sem_presenca = df[f"TP_PRESENCA_{area}"] != 1
    com_nota = df[f"NU_NOTA_{area}"].notna()
    print(area, "- linhas com nota e presença diferente de 1:", (sem_presenca & com_nota).sum())

In [ ]:

print(
    df.groupby("TP_STATUS_REDACAO")["NU_NOTA_REDACAO"]
    .agg(["count", "mean", "min", "max"])
    .round(1)
)
print()


so_status_1 = df[df["TP_STATUS_REDACAO"] == 1]["NU_NOTA_REDACAO"]
todas_com_status = df[df["TP_STATUS_REDACAO"].notna()]["NU_NOTA_REDACAO"]
print("Redação, só status 1:", round(so_status_1.mean(), 1))
print("Redação, todas com status (zeros incluídos):", round(todas_com_status.mean(), 1))
print()


for area in areas:
    presentes = df[df[f"TP_PRESENCA_{area}"] == 1]
    media = round(presentes[f"NU_NOTA_{area}"].mean(), 1)
    vazias = presentes[f"NU_NOTA_{area}"].isna().sum()
    print(area, "| presentes:", len(presentes), "| média:", media, "| notas vazias:", vazias)
print()


presentes_lc = df[df["TP_PRESENCA_LC"] == 1]
print("Com escola informada:", round(presentes_lc["TP_DEPENDENCIA_ADM_ESC"].notna().mean(), 3))
print(presentes_lc["TP_DEPENDENCIA_ADM_ESC"].value_counts(normalize=True).sort_index().round(3))
print(
    "UF da escola vazia, mas dependência preenchida:",
    (df["SG_UF_ESC"].isna() & df["TP_DEPENDENCIA_ADM_ESC"].notna()).sum(),
)

In [9]:
UF_REGIAO = {
    "AC": "Norte", "AM": "Norte", "AP": "Norte", "PA": "Norte",
    "RO": "Norte", "RR": "Norte", "TO": "Norte",
    "AL": "Nordeste", "BA": "Nordeste", "CE": "Nordeste", "MA": "Nordeste",
    "PB": "Nordeste", "PE": "Nordeste", "PI": "Nordeste", "RN": "Nordeste",
    "SE": "Nordeste",
    "DF": "Centro-Oeste", "GO": "Centro-Oeste", "MS": "Centro-Oeste",
    "MT": "Centro-Oeste",
    "ES": "Sudeste", "MG": "Sudeste", "RJ": "Sudeste", "SP": "Sudeste",
    "PR": "Sul", "RS": "Sul", "SC": "Sul",
}

print("UFs que não estão no dicionário:", set(df["SG_UF_PROVA"].unique()) - set(UF_REGIAO))

df["REGIAO"] = df["SG_UF_PROVA"].map(UF_REGIAO)


def tabela_medias(dados, coluna_grupo):
    colunas = {}
    for area in areas:
        presentes = dados[dados[f"TP_PRESENCA_{area}"] == 1]
        colunas[area] = presentes.groupby(coluna_grupo)[f"NU_NOTA_{area}"].mean()
    com_status = dados[dados["TP_STATUS_REDACAO"].notna()]
    colunas["RED_com_zeros"] = com_status.groupby(coluna_grupo)["NU_NOTA_REDACAO"].mean()
    validas = dados[dados["TP_STATUS_REDACAO"] == 1]
    colunas["RED_status1"] = validas.groupby(coluna_grupo)["NU_NOTA_REDACAO"].mean()
    colunas["n_LC"] = dados[dados["TP_PRESENCA_LC"] == 1].groupby(coluna_grupo).size()
    colunas["n_CN"] = dados[dados["TP_PRESENCA_CN"] == 1].groupby(coluna_grupo).size()
    return pd.DataFrame(colunas)


medias_regiao = tabela_medias(df, "REGIAO")
medias_uf = tabela_medias(df, "SG_UF_PROVA")

print(medias_regiao.round(1).sort_values("LC", ascending=False).to_string())
print()
print(medias_uf.round(1).sort_values("LC", ascending=False).to_string())

UFs que não estão no dicionário: set()
                 CN     CH     LC     MT  RED_com_zeros  RED_status1     n_LC     n_CN
REGIAO                                                                                
Sudeste       513.9  530.1  550.9  551.6          609.8        632.7  1168120  1102581
Sul           512.3  528.5  549.8  548.3          592.1        620.3   354680   327669
Centro-Oeste  504.7  514.9  535.4  524.0          593.2        624.4   271902   253159
Nordeste      489.9  495.8  518.3  497.2          563.0        613.5  1285359  1225462
Norte         476.2  485.9  502.0  471.0          532.3        585.0   377494   351465

                CN     CH     LC     MT  RED_com_zeros  RED_status1    n_LC    n_CN
SG_UF_PROVA                                                                        
SP           517.5  534.0  556.6  560.3          607.3        627.1  544811  514500
DF           519.7  534.5  555.2  548.0          609.8        635.6   60673   56958
SC           51